In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2


In [2]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "ORP"] # 6 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

N_COMPONENTS = 4

In [3]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 5):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}"]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

# RNA

In [ ]:
def PrepareData(OriginalDataset, target):
    X_orig = OriginalDataset[PREDICTORS].values
    Y_orig = OriginalDataset[target].values
    
    X_train, X_test, Y_train, Y_test = train_test_split(
        X_orig, Y_orig, test_size=0.2, random_state=42
    )

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)
    
    return x_train, x_test, Y_train, Y_test

    
def PrintDim(x, y):
    print(f"Dimensão da entrada: {np.shape(x)}")
    print(f"Dimensão da saida: {np.shape(y)}")

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

def TrainANN(
    x_train, y_train,
    x_test, y_test,
    n_hidden=[],
    lr=1e-3,
    l2_reg=1e-4,
    epochs=500,
    batch_size=8,
    verbose=0
):
    n_inputs = x_train.shape[1]

    # ======================
    # Modelo
    # ======================
    model = Sequential()

    model.add(
        Dense(
            n_hidden[0],
            activation="relu",
            kernel_regularizer=l2(l2_reg),
            input_shape=(n_inputs,)
        )
    )

    for units in n_hidden[1:]:
        model.add(
            Dense(
                units,
                activation="relu",
                kernel_regularizer=l2(l2_reg)
            )
        )

    model.add(Dense(1, activation="linear"))

    w0 = model.get_weights()


    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss="mse"
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=30,
        restore_best_weights=True
    )

    # ======================
    # Treinamento
    # ======================
    history = model.fit(
        x_train, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=verbose
    )
    wf = model.get_weights()

    # ======================
    # Predições (NORMALIZADAS)
    # ======================
    y_train_pred_norm = model.predict(x_train, verbose=0)
    y_test_pred_norm  = model.predict(x_test,  verbose=0)

    # ======================
    # DESNORMALIZAÇÃO
    # ======================
    y_train_real = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_test_real  = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()

    y_train_pred = OUT_SCALER.inverse_transform(y_train_pred_norm).ravel()
    y_test_pred  = OUT_SCALER.inverse_transform(y_test_pred_norm).ravel()

    # ======================
    # MÉTRICAS NO ESPAÇO FÍSICO
    # ======================
    metrics = {
    "mse_train": round(mean_squared_error(y_train_real, y_train_pred), 4),
    "mse_test":  round(mean_squared_error(y_test_real,  y_test_pred), 4),
    "r2_train":  round(r2_score(y_train_real, y_train_pred), 4),
    "r2_test":   round(r2_score(y_test_real,  y_test_pred), 4)
    } 

    return model, history, metrics, w0, wf


In [ ]:
neurons = [[1], [2], [4], [6], [8], [10], [14], [16], [18], [20]]
all_metrics = []

for i, Dataset in enumerate(Datasets):
    
    for j, target in enumerate(TARGETS):
        
        print(f" → {target}")
        output_dir = f"./Dados/VirtualData/P{i+1}"
        x_train, x_test, y_train, y_test = PrepareData(Dataset, target)
        
        y_train = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
        y_test  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()
        PrintDim(x_train, y_train)
        PrintDim(x_test, y_test)
        
        for neuron in neurons:
            for k in range(10):
                model, history, metrics, w0, wf = TrainANN(
                    x_train, y_train,
                    x_test, y_test,
                    n_hidden=neuron
                )

                # Apenas adiciona metadados
                metrics.update({
                    "P": i + 1,
                    "runTime": k,
                    "target": target,
                    "neurons": neuron,
                    "W0": str([w.round(4).tolist() for w in w0]),
                    "Wf": str([w.round(4).tolist() for w in wf]),
                })

                all_metrics.append(metrics)
                print(metrics)
                # break
        break
    # break



 → Fe
Dimensão da entrada: (20, 6)
Dimensão da saida: (20,)
Dimensão da entrada: (6, 6)
Dimensão da saida: (6,)


{'mse_train': 196325.5175, 'mse_test': 99115.4985, 'r2_train': -0.0456, 'r2_test': 0.3045, 'P': 1, 'runTime': 0, 'target': 'Fe', 'neurons': [1], 'W0': '[[[-0.7968000173568726], [-0.5791000127792358], [-0.004800000227987766], [0.13030000030994415], [0.5023999810218811], [0.6341999769210815]], [0.0], [[-0.8871999979019165]], [0.0]]', 'Wf': '[[[-0.6927000284194946], [-0.5742999911308289], [-0.10890000313520432], [0.24490000307559967], [0.5073999762535095], [0.5527999997138977]], [-0.10419999808073044], [[-0.7838000059127808]], [0.09480000287294388]]'}
{'mse_train': 162397.6586, 'mse_test': 50842.1085, 'r2_train': 0.1351, 'r2_test': 0.6432, 'P': 1, 'runTime': 1, 'target': 'Fe', 'neurons': [1], 'W0': '[[[-0.5479000210762024], [0.0471000000834465], [0.22689999639987946], [0.8123999834060669], [0.6466000080108643], [-0.7822999954223633]], [0.0], [[1.0738999843597412]], [0.0]]', 'W

FileNotFoundError: [Errno 2] No such file or directory: './Dados/VirtualData/P2\\virtual_samples_Fe.xlsx'

In [11]:
results = pd.DataFrame(all_metrics)
print(results)
with pd.ExcelWriter("Results-no-vs.xlsx", engine="openpyxl") as writer:
    results.to_excel(writer, index=False, sheet_name="no-Vs")

      mse_train     mse_test  r2_train  r2_test  P  runTime target neurons  \
0   196325.5175   99115.4985   -0.0456   0.3045  1        0     Fe     [1]   
1   162397.6586   50842.1085    0.1351   0.6432  1        1     Fe     [1]   
2   131780.5856  109506.0310    0.2982   0.2316  1        2     Fe     [1]   
3   192943.6427  146082.5099   -0.0276  -0.0250  1        3     Fe     [1]   
4   193707.9450  174026.9992   -0.0317  -0.2211  1        4     Fe     [1]   
..          ...          ...       ...      ... ..      ...    ...     ...   
95   79663.8101   31743.7480    0.5757   0.7773  1        5     Fe    [20]   
96   78334.5326   43297.7513    0.5828   0.6962  1        6     Fe    [20]   
97   80672.5058   78812.2566    0.5704   0.4470  1        7     Fe    [20]   
98   34289.6567   32897.2927    0.8174   0.7692  1        8     Fe    [20]   
99   52031.3106   43332.9144    0.7229   0.6959  1        9     Fe    [20]   

                                                   W0  \
0   [[